# De Novo Lead-Binding Protein Design — Lead Sensing Project

```
RFD3  →  ProteinMPNN  →  RF3
```

Design *de novo* protein scaffolds that coordinate Pb²⁺ through a Cys3 triad motif
extracted from **PbrR691** (PDB: 5GPE, *Cupriavidus metallidurans* CH34), independent
of DNA binding or protein dimerization.

**Why PbrR691?**  
PbrR691 exhibits **>1000-fold selectivity for Pb²⁺** over competing divalent metal ions
(Chen et al., 2005). This selectivity arises from a unique **hemidirected Pb-S3
coordination geometry** at the homodimer interface, where three conserved cysteine
residues (Cys78, Cys113, Cys122 in 5GPE) form a constrained coordination environment
that Pb²⁺ fits perfectly but other metals cannot (Huang et al., 2016). PbrR691 is the
only Pb-responsive metalloregulator with an experimentally resolved Pb²⁺-bound crystal
structure, making it the ideal starting point for de novo design.

**The design problem:**  
The native PbrR691 signal is generated through Pb²⁺-dependent distortion of promoter DNA —
a complex mechanism requiring homodimerization and a DNA reporter system. We bypass this
by embedding the Pb-Cys3 motif in a *de novo* scaffold that coordinates Pb²⁺ directly,
without DNA. Previous work shows PbrR-derived motifs remain functional outside their
native context (Chen 2005, Wang 2024, Wei 2022), but the minimal structural requirements
for preserving selectivity are unknown. This design campaign tests whether the rigid
hemidirected geometry can be maintained in a completely new protein fold.

**Applications:**
- **Water/wastewater remediation** — surface-immobilized scaffold (via SpyTag+SpyCatcher)
  for selective Pb²⁺ capture from contaminated water.
- **Extracorporeal dialysis** — longer-term direction: an affinity column for circulating
  Pb²⁺ removal in acute/severe lead poisoning cases where chelation therapy is contraindicated
  (renal impairment, nephrotoxicity risk). Unlike chelators, a protein-based column avoids
  systemic metal redistribution and off-target depletion of essential metals (Zn²⁺, Cu²⁺, Fe³⁺).

**Pipeline overview:**
1. **RFD3** — generate scaffold backbones (80–140 aa) steered toward the Pb-Cys3 hotspots
   of 5GPE chains C and D, with Cys sidechain geometry frozen.
2. **ProteinMPNN** — design amino-acid sequences onto each backbone (scaffold chain A only;
   motif Cys geometry preserved).
3. **SpyTag append** — concatenate `GSGSGSAHIVMVDAYKPTK` to each MPNN sequence for
   membrane anchoring.
4. **RF3** — fold and score each construct (scaffold + SpyTag + motif context + Pb²⁺)
   to estimate interface confidence.

Everything runs on the GPU cluster via the **`c27666`** LSF queue.

---
**Motif details** (extracted by `scripts/1a_extract_motif_by_radius_literature_score.py`):
- Source structure: `data/raw/5gpe.cif` — PbrR691 (PDB: 5GPE), 8-chain homodimer
- Selected Pb site: PB_1 (`geometry_call = likely_hemidirected_pb_s3`)
- Pb-Cys3 triad (Pb-S distances from EXAFS/crystal; target 2.67 ± 0.15 Å):
  - **C78** (chain C) — Pb-S = 2.60 Å (one Cys from the 'C' protomer)
  - **D113** (chain D) — Pb-S = 2.67 Å (first Cys from the 'D' protomer)
  - **D122** (chain D) — Pb-S = 2.73 Å (second Cys from the 'D' protomer)
- Context fed to RFD3: C72–84 (13 res) and D107–128 (22 res) — 6-residue window each side

## How to run this notebook

**Read `README.md` first** for full setup. In short:

1. This notebook lives in `design/` inside your project clone.
   Run it from there — `REPO_ROOT` is auto-detected as the notebook's directory.
2. Register the shared conda env as a Jupyter kernel once (see README), then select it.
3. Run cells top to bottom. GPU stages **do not run in the notebook** — each writes an
   LSF submit script and prints a `bsub < ...` command. Run that in a terminal, wait
   for the job to finish (`bstat`), then continue with the next cell.

**Stage order:**  
RFD3 (submit → wait) → process metrics → filter → MPNN (submit → wait) → check FASTAs
→ RF3 (build JSONs → submit → wait) → score → collect best binders → [optional] tandem repeats.

**Queue notes (`c27666`):** shared on a few GPUs (~20 GB MIG slices). Keep design counts
small (defaults are tiny). Max wall time 12 h; the queue default is 15 min —
every submit script sets `-W` explicitly. Monitor with `bstat`; kill with `bkill <jobid>`.

## Setup: imports, repo paths, working directories

In [12]:
# --- Core ---
import os, sys, re, glob, json, math, shutil, csv
from pathlib import Path
from copy import deepcopy

# --- Data / plotting ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Locate the repo root (the design/ folder containing this notebook) ---
REPO_ROOT = Path.cwd()
if not ((REPO_ROOT / "inputs").is_dir() and (REPO_ROOT / "lib").is_dir()):
    raise RuntimeError(
        f"cwd={REPO_ROOT} is not the repo root (no inputs/ and lib/).\n"
        "Set it manually: REPO_ROOT = Path('/dtu/blackhole/09/214281/lead_sensoring/design')"
    )

# --- Lead-sensing project inputs (pre-computed by 00_motif_scan.sh) ---
LEAD_INPUTS = REPO_ROOT / "lead_inputs"
if not LEAD_INPUTS.is_dir():
    raise RuntimeError(
        f"lead_inputs/ not found under {REPO_ROOT}. "
        "Run 00_motif_scan.sh first to generate the motif inputs."
    )

# --- Key input files ---
# Full 5GPE CIF structure (PbrR691 homodimer with Pb2+; RFD3 input)
STRUCTURE_CIF = (REPO_ROOT.parent / "data" / "raw" / "5gpe.cif").resolve()
# Reduced context PDB: chains C (res 72-84) + D (res 107-128) + Pb2+ atom only
# This is the trimmed target used for RF3 folding (generated by 1c_export_rfd3_context_structure.py)
CONTEXT_PDB = (LEAD_INPUTS / "rfd3_context_fixed_segments_plus_pb.pdb").resolve()

for p in [STRUCTURE_CIF, CONTEXT_PDB]:
    if not p.exists():
        print(f"[WARNING] Not found: {p}")

# --- Helper modules (note: nested lib/lib/ in this repo clone) ---
sys.path.insert(0, str(REPO_ROOT / "lib" / "lib"))
import jupyter_utils
from rf3_metrics import gather_rf3_metrics

print("Repo root  :", REPO_ROOT)
print("Lead inputs:", LEAD_INPUTS)
print("5GPE CIF   :", STRUCTURE_CIF)
print("Context PDB:", CONTEXT_PDB)
print("User       :", os.environ.get("USER", "?"))

Repo root  : /dtu/blackhole/09/214281/lead_sensoring/design
Lead inputs: /dtu/blackhole/09/214281/lead_sensoring/design/lead_inputs
5GPE CIF   : /dtu/blackhole/09/214281/lead_sensoring/data/raw/5gpe.cif
Context PDB: /dtu/blackhole/09/214281/lead_sensoring/design/lead_inputs/rfd3_context_fixed_segments_plus_pb.pdb
User       : s243548


In [13]:
# All run outputs go under work/<experiment>/ inside the design/ folder (git-ignored).
experiment = "exp_01"          # bump for each new design round
WORK = REPO_ROOT / "work" / experiment

_subdirs = ["cmds", "submit", "logs", "configs", "scores",
            "diffusion_out", "mpnn_out", "rf3_out", "best_binders"]
for d in _subdirs:
    (WORK / d).mkdir(parents=True, exist_ok=True)

cmds_dir          = str(WORK / "cmds")
submit_dir        = str(WORK / "submit")
logs_dir          = str(WORK / "logs")
configs_dir       = str(WORK / "configs")
scores_dir        = str(WORK / "scores")
diffusion_out_dir = str(WORK / "diffusion_out")
mpnn_out_dir      = str(WORK / "mpnn_out")
rf3_out_dir       = str(WORK / "rf3_out")
best_binders_dir  = str(WORK / "best_binders")

# --- SpyTag for C-terminal membrane anchoring (via SpyCatcher) ---
# SpyCatcher-coated membrane + this tag = covalent attachment via spontaneous isopeptide bond.
# The GSGSGS linker decouples the rigid Pb-binding scaffold from the tag,
# keeping the tag surface-exposed for SpyCatcher conjugation.
# This sequence is appended AFTER MPNN designs the scaffold (before RF3 validation).
SPYTAG_LINKER = "GSGSGS"          # 6 aa GS-repeat flexible linker
SPYTAG_SEQ    = "AHIVMVDAYKPTK"   # 13 aa SpyTag (Baker lab, Zakeri et al. 2012)
SPYTAG        = SPYTAG_LINKER + SPYTAG_SEQ   # 19 aa total C-terminal extension

pd.set_option("display.max_columns", None)
print("Working dir:", WORK)
print("SpyTag     :", SPYTAG, f"({len(SPYTAG)} aa)")

Working dir: /dtu/blackhole/09/214281/lead_sensoring/design/work/exp_01
SpyTag     : GSGSGSAHIVMVDAYKPTK (19 aa)


## Scientific background and design rationale

### Why the Cys geometry must be preserved exactly

The extraordinary Pb²⁺ selectivity of PbrR691 arises from a **hemidirected coordination
geometry**: the three Cys SG atoms (from C78, D113, D122) are positioned on one side of
the metal, with the opposite side sterically open. This arrangement accommodates Pb²⁺'s
unique 6s² lone pair (which dictates hemidirected geometry) but is incompatible with
Zn²⁺, Cd²⁺, or other competing metals that prefer symmetric (holodirected) coordination.

Pb-S distances of ~2.67 Å (EXAFS) and the specific S-Pb-S angles are critical — even
small deviations reduce selectivity. This is why we set `redesign_motif_sidechains: false`
in RFD3: the crystallographic Cys coordinates are frozen, preserving the exact geometry
that evolution optimized.

### Design strategy: binder-style scaffolding

In native PbrR691, the Pb-Cys3 site sits at the interface of a homodimer (chain C + chain D).
Our approach treats those dimer interface segments (C72–84 and D107–128) as a **pseudo-target**
and uses RFD3 in binder-design mode to generate a *de novo* scaffold that wraps around them:

```
RFD3 output (3 chains):
  Chain A  — new scaffold (80–140 aa, the designed protein)
  Chain C  — frozen segment from 5GPE (res 72–84, contains Cys78)
  Chain D  — frozen segment from 5GPE (res 107–128, contains Cys113, Cys122)
  + Pb²⁺   — held in place by the frozen Cys triad
```

Chain A is the actual product — a single-chain protein that contacts all three Cys
residues and thereby holds the Pb²⁺ coordination site in the correct geometry without
needing a partner protomer or DNA.

### Multiple Pb-binding sites and sensor sensitivity

Avidity matters for low-concentration detection and for affinity columns. A tandem repeat
of the scaffold (two copies linked by GSGSGSG) provides two independent Pb²⁺ sites per
chain, doubling capture capacity without requiring co-expression of multiple chains.
The SpyTag at the C-terminus anchors the tandem protein on the membrane in a defined
orientation. See the optional tandem-repeat cell at the end of this notebook.

### Key literature

| Ref | Finding relevant to this project |
|-----|-----------------------------------|
| Chen et al., *Angew. Chem.* 2005 | PbrR691 shows >1000-fold selectivity; first fluorescent Pb²⁺ reporter via DNA distortion |
| Huang et al., *Inorg. Chem.* 2016 | Crystal structure of 5GPE; hemidirected Pb-S3 geometry at dimer interface |
| Wang et al., *Anal. Chim. Acta* 2024 | PbrR-derived peptide fused to sfGFP detects Pb²⁺ in serum — motif works outside native context |
| Wei et al., *Polym. Chem.* 2022 | PbrR-derived peptide in PNIPAM hydrogel for reversible Pb²⁺ capture — rigidity is key |
| Zakeri et al., *PNAS* 2012 | SpyTag/SpyCatcher isopeptide bond system for irreversible surface attachment |

## 1. RFD3 — generate scaffold backbones around the Pb-Cys3 motif

RFD3 diffuses a new protein backbone from noise, guided by the frozen Pb-Cys3 motif
segments from PbrR691 (5GPE). Two parameters define the design:

**`contig`** — what to build:
- `80-140` — free scaffold (the new de novo protein, chain A in output; length sampled per design)
- `/0` — chain break separating the scaffold from the fixed target context
- `C72-84` — fixed 13-residue segment from chain C (contains Cys78, one Pb ligand)
- `/0,D107-128` — fixed 22-residue segment from chain D (contains Cys113 and Cys122)

**`select_hotspots`** — the three Cys SG atoms that coordinate Pb²⁺. RFD3's `hotspots`
origin strategy places the scaffold origin relative to these atoms and steers the backbone
to contact them. This is what drives the new scaffold to engage the Pb coordination site.

**`redesign_motif_sidechains: false`** — the crystallographic Cys sidechain coordinates
from 5GPE are frozen. This is essential: the hemidirected Pb-S3 geometry that gives PbrR691
its >1000-fold selectivity depends on exact Pb-S distances (~2.67 Å) and S-Pb-S angles.
Even small distortions to the Cys positions would compromise selectivity.

**`is_non_loopy: true`** — biases RFD3 toward helical secondary structure. More ordered
scaffolds tend to produce better MPNN sequences and higher RF3 pLDDT.

**`inference_sampler.step_scale=3` and `gamma_0=0.2`** — lower-temperature diffusion;
produces more structured, designable backbones (recommended by RFD3 docs for PPI design).

---
**Multi-site strategy:** This campaign designs a scaffold with ONE Pb-binding site per chain.
For two Pb²⁺ sites, duplicate the best MPNN sequence with a GSGSGSG linker (see the optional
tandem-repeat cell at the end). Do NOT add a second copy of the motif context to the contig
here — in 5GPE both copies would have identical coordinates, which confuses RFD3.
The tandem approach is cleaner and avoids this.

### Build the RFD3 input JSON

This cell generates the RFD3 input JSON. The resulting file is equivalent to
`lead_inputs/rfd3_input_radius_4p0.json` pre-computed by the motif scan pipeline,
but with the correct absolute path for this clone's 5GPE CIF.

In [14]:
design_name = "5gpe_pb_motif_r4"

# RFD3 input: pre-trimmed context PDB with only the motif segments and Pb2+.
# We use this instead of the full 5gpe.cif because inference_load_() builds the
# biological assembly from CIF files, which remaps chains A-H → A, B, I, J —
# eliminating chains C and D that the contig references.
# rfd3_input_context.pdb has: chain C (res 72-84), chain D (res 107-128),
# chain E (Pb2+, moved off chain C to avoid chain-overlap error).
input_pdb = str(LEAD_INPUTS / "rfd3_input_context.pdb")

# ── Contig ──────────────────────────────────────────────────────────────────────
# 80-140   : free scaffold (de novo protein, chain A; length sampled per backbone)
# /0       : chain break between scaffold and fixed context
# C72-84   : fixed 13-residue segment from chain C (window-6 around Cys78 at pos 78)
# /0       : chain break
# D107-128 : fixed 22-residue segment from chain D (window-6 around Cys113 and Cys122)
#
# Window of 6 residues was chosen by scripts/1b_make_rfdiffusion_inputs.py
# (--target-window 6), providing enough backbone context for RFD3 to model the
# scaffold contacts without including unnecessary flanking residues.
contig = "80-140,/0,C72-84,/0,D107-128"

# Total design length = scaffold(80-140) + C-segment(13) + D-segment(22) = 115-175
length = "115-175"

# Critical: freeze Cys sidechain coordinates to preserve the hemidirected Pb-S3
# coordination geometry that underlies PbrR691's >1000-fold Pb2+ selectivity.
redesign_motif_sidechains = False

# ── Hotspots ────────────────────────────────────────────────────────────────────
# Only the Cys SG atoms — these are the atoms that directly ligate Pb2+.
# Using SG-only (not the full residue) gives RFD3 atomic-level precision when
# placing the scaffold origin and steering contacts.
select_hotspots = {
    "C78":  "SG",   # Cys78, chain C — Pb-S = 2.60 Å in 5GPE
    "D113": "SG",   # Cys113, chain D — Pb-S = 2.67 Å in 5GPE
    "D122": "SG",   # Cys122, chain D — Pb-S = 2.73 Å in 5GPE
}

infer_ori_strategy = "hotspots"   # place scaffold origin relative to the Cys triad centroid
is_non_loopy       = True          # bias toward helices; fewer loops → better designability

rfd3_json = str(Path(configs_dir) / "rfd3_input.json")

payload = {
    design_name: {
        "dialect": 2,
        "input": input_pdb,
        "contig": contig,
        "length": length,
        "redesign_motif_sidechains": redesign_motif_sidechains,
        "select_hotspots": select_hotspots,
        "infer_ori_strategy": infer_ori_strategy,
        "is_non_loopy": is_non_loopy,
    }
}
with open(rfd3_json, "w") as f:
    json.dump(payload, f, indent=2)

print("Wrote RFD3 input ->", rfd3_json)
print("Input PDB  :", input_pdb)
print("Design name:", design_name)
print("Contig     :", contig)
print("Length     :", length)
print("Hotspots   :", select_hotspots)
print()
print("Pre-computed reference contig (should match):")
ref = LEAD_INPUTS / "rfd3_input_radius_4p0.json"
if ref.exists():
    with open(ref) as f:
        ref_data = json.load(f)
    ref_contig = list(ref_data.values())[0]["contig"]
    print(f"  Reference contig: {ref_contig}")
    print(f"  Contig match: {ref_contig == contig}")

Wrote RFD3 input -> /dtu/blackhole/09/214281/lead_sensoring/design/work/exp_01/configs/rfd3_input.json
Input PDB  : /dtu/blackhole/09/214281/lead_sensoring/design/lead_inputs/rfd3_input_context.pdb
Design name: 5gpe_pb_motif_r4
Contig     : 80-140,/0,C72-84,/0,D107-128
Length     : 115-175
Hotspots   : {'C78': 'SG', 'D113': 'SG', 'D122': 'SG'}

Pre-computed reference contig (should match):
  Reference contig: 80-140,/0,C72-84,/0,D107-128
  Contig match: True


### Write the RFD3 submit script

Default: `diffusion_batch_size=1 × n_batches=4` → **4 backbones** total — deliberately
small to respect the shared queue. For a production run, increase `n_batches` to 50–200
and extend `-W` accordingly. **Always check `bqueues c27666` before scaling up.**

Key RFD3 hyperparameters:
- `inference_sampler.step_scale=3` and `gamma_0=0.2` — lower-temperature sampling;
  produces more structured backbones that are easier to sequence-design downstream.
  These settings are specifically recommended for protein–protein interface design.

In [15]:
queue        = "c27666"
job_name     = "rfd3_pb"
cores        = 4
gpu_spec     = "num=1:mode=exclusive_process"
time_limit   = "1:00"
mem          = "10GB"

# Model weights (local blackhole copy; same checkpoint as /dtu/projects/dbl/foundry/ckpt/)
CKPT_PATH = "/dtu/blackhole/00/c27666/27666_Protein_Design/weights/rfd3_latest.ckpt"

diffusion_batch_size = 1    # backbones per GPU batch
n_batches            = 4    # total backbones = diffusion_batch_size * n_batches

script_path = os.path.join(submit_dir, "rfd3.sh")

script = f"""#!/bin/sh
#BSUB -q {queue}
#BSUB -J {job_name}
#BSUB -n {cores}
#BSUB -gpu "{gpu_spec}"
#BSUB -W {time_limit}
#BSUB -R "rusage[mem={mem}]"
#BSUB -R "span[hosts=1]"
#BSUB -o {logs_dir}/%J.out
#BSUB -e {logs_dir}/%J.err

mkdir -p {logs_dir} {diffusion_out_dir}
module load cuda/12.4
source /dtu/blackhole/00/c27666/miniforge3/etc/profile.d/conda.sh
conda activate /dtu/blackhole/00/c27666/miniforge3/envs/protein-design

# Workarounds for MIG GPU slices on c27666 A100s
export DISABLE_CUEQUIVARIANCE=true           # skip cuEquivariance (crashes on MIG via pynvml)
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True  # reduce fragmentation on ~20GB slice

rfd3 design \\
    out_dir="{diffusion_out_dir}" \\
    inputs="{rfd3_json}" \\
    ckpt_path="{CKPT_PATH}" \\
    diffusion_batch_size={diffusion_batch_size} \\
    n_batches={n_batches} \\
    low_memory_mode=True \\
    inference_sampler.step_scale=3 \\
    inference_sampler.gamma_0=0.2

echo "Completed at $(date)"
"""

with open(script_path, "w") as f:
    f.write(script)

print("Wrote", script_path)
print(f"\nThis will generate {diffusion_batch_size * n_batches} backbone(s).")
print("\nSubmit in a terminal:")
print("  bsub < " + script_path)
print("Monitor: bstat   (job disappears when done)")
print("Logs   :", logs_dir)

Wrote /dtu/blackhole/09/214281/lead_sensoring/design/work/exp_01/submit/rfd3.sh

This will generate 4 backbone(s).

Submit in a terminal:
  bsub < /dtu/blackhole/09/214281/lead_sensoring/design/work/exp_01/submit/rfd3.sh
Monitor: bstat   (job disappears when done)
Logs   : /dtu/blackhole/09/214281/lead_sensoring/design/work/exp_01/logs


### Process RFD3 outputs

Each backbone is written as a `.cif.gz` plus a `.json` of geometry metrics. Run the cells
below **after** the RFD3 job finishes to gather and inspect those metrics.

Key metrics to watch:
- **`helix_fraction`** — fraction of residues in helical SS; higher is better for well-packed
  stable designs (especially with `is_non_loopy=True`). Aim for > 0.6.
- **`loop_fraction`** — ideally < 0.4 for designable backbones.
- **`n_chainbreaks`** — should be **2** (one per `/0` in the contig: scaffold A + context C + context D).
- **`n_clashing.*`** — both should be 0 (any clashes indicate a failed backbone).
- **`radius_of_gyration`** — compactness proxy; very large values suggest extended/unfolded backbones.

In [ ]:
json_rfd3_dir    = diffusion_out_dir
rfd3_metrics_csv = Path(scores_dir) / "rfd3_metrics_with_json_path.csv"

json_paths = sorted(glob.glob(os.path.join(json_rfd3_dir, "*.json")))
if not json_paths:
    raise SystemExit(f"No JSON files found in {json_rfd3_dir!r} — run RFD3 first.")

rows = []
all_keys = set(["json_path"])

for jp in json_paths:
    with open(jp, "r") as f:
        data = json.load(f)

    metrics = data.get("metrics", {})
    row = {"json_path": os.path.abspath(jp)}
    for k, v in metrics.items():
        if k in ("diffused_com", "fixed_com") and isinstance(v, (list, tuple)) and len(v) == 3:
            row[f"{k}_x"], row[f"{k}_y"], row[f"{k}_z"] = v
            all_keys.update({f"{k}_x", f"{k}_y", f"{k}_z"})
        else:
            row[k] = v
            all_keys.add(k)
    rows.append(row)

fieldnames = ["json_path"] + sorted(k for k in all_keys if k != "json_path")

with open(rfd3_metrics_csv, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for r in rows:
        writer.writerow(r)

print(f"Wrote {len(rows)} rows → {rfd3_metrics_csv}")
print("Columns:", fieldnames)

In [ ]:
df = pd.read_csv(rfd3_metrics_csv)
df.head(10)

In [ ]:
metrics = [
    "alanine_content",
    "glycine_content",
    "helix_fraction",
    "loop_fraction",
    "sheet_fraction",
    "max_ca_deviation",
    "n_chainbreaks",
    "n_clashing.interresidue_clashes_w_backbone",
    "n_clashing.interresidue_clashes_w_sidechain",
    "non_loop_fraction",
    "radius_of_gyration",
]

metrics = [m for m in metrics if m in df.columns]
print("Plotting:", metrics)

n_cols = 4
n_rows = math.ceil(len(metrics) / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows))
axes = axes.flatten()

def plot_hist(ax, series, title):
    s = series.dropna()
    if s.empty:
        ax.set_title(f"{title} (no data)")
        ax.axis("off")
        return
    ax.hist(s, bins=30)
    ax.set_title(title)
    ax.set_ylabel("count")

for ax, col in zip(axes, metrics):
    plot_hist(ax, df[col], col)
for ax in axes[len(metrics):]:
    ax.axis("off")

plt.suptitle("RFD3 backbone metrics — Pb-Cys3 scaffold campaign", y=1.02)
plt.tight_layout()
plt.show()

### Optional: filter backbones by secondary structure content

Backbones with too much loop tend to produce less stable designs downstream.
Since we set `is_non_loopy=True`, most backbones should pass the `loop_fraction < 0.4`
threshold. Passing designs are copied to `passing_loop_nonloop/` so MPNN picks them up;
if none pass (very small batch), MPNN falls back to the full set.

Skip this section to keep **all** backbones.

In [ ]:
df_filt = df[
    (df["loop_fraction"] < 0.4) &
    (df["non_loop_fraction"] > 0.6)
].copy()

print(f"Total designs  : {len(df)}")
print(f"Passing designs: {len(df_filt)}")
df_filt.head()

In [ ]:
PATH_COL = "json_path"
filter_name = "passing_loop_nonloop"

filtered_dir = Path(diffusion_out_dir) / filter_name
filtered_dir.mkdir(parents=True, exist_ok=True)

def copy_design(json_path: str, dst_dir: Path) -> int:
    """Copy one design's JSON and its matching CIF/CIF.GZ into dst_dir."""
    jp = Path(json_path)
    if not jp.exists():
        return 0
    copied = 0
    dst_json = dst_dir / jp.name
    if not dst_json.exists():
        shutil.copy2(jp, dst_json)
        copied += 1
    stem = jp.stem
    for cif in (jp.with_name(stem + ".cif"), jp.with_name(stem + ".cif.gz")):
        if cif.exists():
            dst_cif = dst_dir / cif.name
            if not dst_cif.exists():
                shutil.copy2(cif, dst_cif)
                copied += 1
    return copied

total_copied = missing = 0
unique_paths = df_filt[PATH_COL].dropna().unique()
for pth in unique_paths:
    if not isinstance(pth, str) or not os.path.exists(pth):
        print(f"[MISSING] {pth}")
        missing += 1
        continue
    total_copied += copy_design(pth, filtered_dir)

print(f"Passing designs: {len(unique_paths)}")
print(f"Copied files   : {total_copied}")
print(f"Output folder  : {filtered_dir}")

## 2. ProteinMPNN — design sequences

ProteinMPNN writes amino-acid sequences onto each RFD3 backbone. We redesign the whole
scaffold chain (`A`) freely; the motif chains (C and D from 5GPE) are fixed — ProteinMPNN
sees them as context but does not redesign them. This preserves the Cys3 triad exactly
as crystallographically observed.

After MPNN, we append the SpyTag (`GSGSGSAHIVMVDAYKPTK`) to every designed sequence
in the RF3 input-building step below.

This builds one command per backbone and submits them as a single LSF **job array**.

In [ ]:
seed              = 42
batch_size        = 2     # sequences per backbone per batch
number_of_batches = 2     # total sequences per backbone = batch_size * number_of_batches
chains_to_design  = "A"   # redesign only the de novo scaffold; C and D are context-only

# ProteinMPNN weights (local blackhole copy)
ckpt = "/dtu/blackhole/00/c27666/27666_Protein_Design/weights/proteinmpnn_v_48_020.pt"

cmds = f"{cmds_dir}/mpnn.cmds"
filtered = sorted(glob.glob(f"{diffusion_out_dir}/passing_loop_nonloop/*.cif.gz"))
structures = filtered if filtered else sorted(glob.glob(f"{diffusion_out_dir}/*.cif.gz"))
print(f"Using {len(structures)} backbone(s) "
      + ("(loop-filtered)" if filtered else "(all backbones — no filter applied)"))
if not structures:
    raise SystemExit(f"No backbones (*.cif.gz) in {diffusion_out_dir} — run RFD3 first.")

with open(cmds, "w") as f:
    for structure in structures:
        bn = os.path.basename(structure).replace(".cif.gz", "")
        this_out = f"{mpnn_out_dir}/{bn}"
        os.makedirs(this_out, exist_ok=True)
        cmd = (f"mpnn --seed {seed} --structure_path \"{structure}\" "
               f"--out_directory \"{this_out}\" --batch_size {batch_size} "
               f"--number_of_batches {number_of_batches} --model_type protein_mpnn "
               f"--checkpoint_path \"{ckpt}\" --is_legacy_weights True "
               f"--designed_chains \"{chains_to_design}\"")
        f.write(cmd + "\n")

n_tasks = sum(1 for _ in open(cmds))
sub_script = jupyter_utils.make_sub_script(
    cmds, n_task=n_tasks, group_size=10, mem="10G", queue="c27666",
    job_name="mpnn_pb", cores=4, time_limit="2:00",
    env="/dtu/blackhole/00/c27666/miniforge3/envs/protein-design",
)
print(f"\n{n_tasks} MPNN command(s) written to: {cmds}")
print(f"Each backbone gets {batch_size * number_of_batches} sequence(s).")
print("Submit:\n  bsub < " + sub_script)

### SpyTag strategy and multiple-site design

**SpyTag** (`GSGSGSAHIVMVDAYKPTK`, 19 aa):
- The 6-residue GSGSGS linker provides conformational flexibility so the Pb-binding
  scaffold and the membrane surface can move independently — critical because the
  rigid Pb-coordination geometry must not be distorted by surface attachment.
- The 13-residue SpyTag forms a **spontaneous isopeptide bond** with SpyCatcher
  (Lys→Asp side chains, no catalyst required, essentially irreversible).
- For sensing: SpyCatcher-coated magnetic beads or membranes capture the protein
  in a defined orientation, pointing the Pb-binding site toward the solution.
- For the dialysis column application: SpyCatcher-functionalized resin immobilizes
  the protein in a packed column for flow-through Pb²⁺ capture from blood/plasma.

The SpyTag is appended to each MPNN sequence **in the RF3 input-building step** below,
so RF3 folds the full construct and we can check that the tag region is surface-accessible.
In the RF3 output, low pLDDT in the last ~19 residues of chain A is **acceptable** —
an unstructured, flexible tag is exactly what allows SpyCatcher binding.

**Multiple Pb-binding sites (tandem repeat approach):**  
Each monomer has one Pb²⁺ binding site. To double the capture capacity:
```
[scaffold_1] — GSGSGSG — [scaffold_2] — GSGSGSAHIVMVDAYKPTK
```
This tandem protein has **two independent Pb-Cys3 sites** plus the SpyTag, all in
one single chain (~200–300 aa total). See the optional cell at the end of this notebook.

Run the cell below **after MPNN finishes** to verify that FASTA files are present.

In [ ]:
# Inspect MPNN output FASTAs (run after MPNN job completes)
fa_files = sorted(glob.glob(os.path.join(mpnn_out_dir, "**", "*.fa"), recursive=True))
print(f"Found {len(fa_files)} FASTA file(s) under {mpnn_out_dir}")
for fa in fa_files[:5]:
    print(" ", fa)
if len(fa_files) > 5:
    print(f"  ... and {len(fa_files)-5} more")
if not fa_files:
    print("No FASTAs found — run MPNN first, then re-run this cell.")

## 3. RF3 — fold and score each construct

RF3 folds the complete construct: **scaffold chain A** (MPNN sequence + SpyTag) together
with the **fixed motif context** (chains C and D from 5GPE, including the Pb²⁺ ion).

This lets us evaluate:
- **Scaffold fold quality** — does chain A fold confidently? (binder pLDDT, 0–1)
- **Interface with Cys78** — is the scaffold–C78 interface well-defined? (PAE, ipSAE vs B_1)
- **Interface with Cys113/122** — same for the D-chain context (PAE, ipSAE vs C_1)
- **SpyTag accessibility** — last ~19 residues of chain A; low pLDDT = disordered = accessible

**Chain ID mapping in RF3 output** (order = components in input JSON):
```
A_1 — de novo scaffold + SpyTag   (our designed protein)
B_1 — chain C from 5GPE, res 72–84    (contains Cys78; one Pb-ligand)
C_1 — chain D from 5GPE, res 107–128  (contains Cys113 + Cys122; two Pb-ligands)
D_1 — Pb²⁺ ion   (HETATM in the context PDB, chain C, res 201)
```
*Verify the actual chain IDs from the first `.score` file after the run — update
`BINDER`, `TARGET_F`, `TARGET_G` below if they differ.*

### Write RF3 template and build RF3 input JSONs

First we write a project-specific RF3 template JSON that specifies:
- **Chain A component** — binder sequence placeholder (filled in per MPNN design + SpyTag)
- **Context path** — `rfd3_context_fixed_segments_plus_pb.pdb` (chains C, D + Pb²⁺ atom)
- **`template_selection`** — tell RF3 to use chains C and D as structural templates

Then one RF3 input JSON is instantiated per MPNN sequence (with SpyTag appended).

In [ ]:
# Write the RF3 template for the lead-sensing project.
# This template is created once per project; the binder sequence is filled per-design below.

rf3_template_path = str(LEAD_INPUTS / "rf3_template_lead.json")

rf3_template = {
    "name": "lead_binder_DESIGNNAME",
    "components": [
        {
            # Chain A: de novo scaffold + SpyTag (sequence filled in per MPNN design)
            "seq": "BINDER_SEQUENCE_HERE",
            "chain_id": "A"
        },
        {
            # Target context: chains C (Cys78 region) + D (Cys113/122 region) + Pb2+ ion
            # Generated by scripts/1c_export_rfd3_context_structure.py from 5gpe.cif.
            # Path is relative to REPO_ROOT and will be resolved to absolute in the next cell.
            "path": "lead_inputs/rfd3_context_fixed_segments_plus_pb.pdb"
        }
    ],
    # Use chains C and D as structural templates for RF3 (provides homology info)
    "template_selection": ["C", "D"]
}

with open(rf3_template_path, "w") as f:
    json.dump(rf3_template, f, indent=2)

print("Wrote RF3 template ->", rf3_template_path)
print()
print(json.dumps(rf3_template, indent=2))

In [ ]:
rf3_json_dir = Path(configs_dir) / "rf3"
rf3_json_dir.mkdir(parents=True, exist_ok=True)

with open(rf3_template_path) as f:
    template = json.load(f)

# Resolve the context PDB to an absolute path so RF3 finds it from any working directory
for comp in template["components"]:
    if isinstance(comp, dict) and "path" in comp:
        comp["path"] = str((REPO_ROOT / comp["path"]).resolve())

# Locate the chain A binder component (has a 'seq' placeholder)
binder_idx = next(
    i for i, c in enumerate(template["components"])
    if isinstance(c, dict) and c.get("chain_id") == "A" and "seq" in c
)

# MPNN FASTA header format: >{name}_b{batch_idx}_d{design_idx}
DESIGN_RE = re.compile(r"_b(\d+)_d(\d+)")

def parse_fasta(path):
    header, chunks = None, []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if header is not None:
                    yield header, "".join(chunks)
                header, chunks = line[1:], []
            else:
                chunks.append(line)
        if header is not None:
            yield header, "".join(chunks)

fa_files = sorted(glob.glob(os.path.join(mpnn_out_dir, "**", "*.fa"), recursive=True))
if not fa_files:
    raise SystemExit(f"No MPNN FASTAs under {mpnn_out_dir} — run MPNN first.")

written = 0
for fa in fa_files:
    model_dir = os.path.basename(os.path.dirname(fa))  # backbone name from parent dir
    for header, seq in parse_fasta(fa):
        m = DESIGN_RE.search(header)
        if not m:
            continue
        batch_idx  = int(m.group(1))
        design_idx = int(m.group(2))

        # Full chain A = MPNN scaffold sequence + SpyTag (GSGSGSAHIVMVDAYKPTK)
        # The SpyTag is appended here so RF3 folds the complete construct and can
        # evaluate whether the tag is surface-accessible (low pLDDT = disordered = good).
        scaffold_seq = seq.split(":", 1)[0]      # scaffold chain A only (before any ':' separator)
        chain_a      = scaffold_seq + SPYTAG      # final sequence: scaffold + GSGSGSAHIVMVDAYKPTK

        j = deepcopy(template)
        j["components"][binder_idx]["seq"] = chain_a
        name = f"{model_dir}__b{batch_idx}_d{design_idx}"
        j["name"] = name
        with open(rf3_json_dir / f"{name}.json", "w") as out:
            json.dump(j, out, indent=2)
        written += 1

print(f"Wrote {written} RF3 input JSONs -> {rf3_json_dir}")
print(f"Chain A length = scaffold ({scaffold_seq.__class__.__name__}?) + SpyTag ({len(SPYTAG)} aa)")
print(f"SpyTag sequence: {SPYTAG}")

### Write the RF3 submit script (job array)

In [ ]:
queue       = "c27666"
job_name    = "rf3_pb"
time_limit  = "2:00"
mem         = "10GB"
gpu_spec    = "num=1:mode=exclusive_process"
cores       = 4
group_size  = 20              # RF3 folds per array task

# RF3 environment (dedicated release; has activate_env.sh)
rf3_release = "/dtu/projects/dbl/rf3/release"
# RF3 checkpoint weights (local blackhole copy; remapped for this cluster)
ckpt_path   = "/dtu/blackhole/00/c27666/27666_Protein_Design/weights/rf3_foundry_01_24_latest_remapped.ckpt"
num_steps   = 50

# RF3 wrapper: patches pynvml for MIG GPU slices (lives in lib/lib/ in this repo)
rf3_wrapper = str(REPO_ROOT / "lib" / "lib" / "rf3_wrapper.py")

json_files = sorted(Path(rf3_json_dir).glob("*.json"))
if not json_files:
    raise SystemExit(f"No RF3 JSONs in {rf3_json_dir} — build them first.")

cmds_path = Path(cmds_dir) / "rf3.cmds"
lines = []
for jf in json_files:
    out_dir = Path(rf3_out_dir) / jf.stem
    lines.append(
        f"python {rf3_wrapper} fold inference_engine=rf3 inputs={jf} out_dir={out_dir} "
        f"ckpt_path={ckpt_path} num_steps={num_steps} "
        f"annotate_b_factor_with_plddt=True early_stopping_plddt_threshold=0"
    )
cmds_path.write_text("\n".join(lines) + "\n")

n_arrays = math.ceil(len(lines) / group_size)
script = f"""#!/bin/sh
#BSUB -q {queue}
#BSUB -J {job_name}[1-{n_arrays}]
#BSUB -n {cores}
#BSUB -gpu "{gpu_spec}"
#BSUB -W {time_limit}
#BSUB -R "rusage[mem={mem}]"
#BSUB -R "span[hosts=1]"
#BSUB -o {logs_dir}/%J_%I.out
#BSUB -e {logs_dir}/%J_%I.err

mkdir -p {logs_dir}
module load cuda/12.4
source {rf3_release}/activate_env.sh
export DISABLE_CUEQUIVARIANCE=true

CMDS_FILE={cmds_path}
GROUP_SIZE={group_size}
START=$(( (LSB_JOBINDEX - 1) * GROUP_SIZE ))
END=$(( START + GROUP_SIZE ))
i=0
while IFS= read -r cmd; do
    if [ "$i" -ge "$START" ] && [ "$i" -lt "$END" ]; then
        echo "Running: $cmd"
        eval "$cmd"
    fi
    i=$((i+1))
done < "$CMDS_FILE"
echo "Done at $(date)"
"""

sub = Path(submit_dir) / "rf3.sh"
sub.write_text(script)
print(f"{len(lines)} RF3 fold(s) in {n_arrays} array task(s).")
print(f"RF3 ckpt  : {ckpt_path}")
print(f"RF3 env   : {rf3_release}/activate_env.sh")
print("Submit:\n  bsub < " + str(sub))

### Score and plot RF3 results

Run after the RF3 jobs finish. Key metrics per design:

| Metric | Meaning | Target |
|--------|---------|--------|
| `best_binder_plddt` | Confidence of chain A fold (0–1) | > 0.85 |
| `AF_best_min_pae` | Interface PAE: scaffold (A_1) ↔ Cys78 context (B_1) in Å | < 5.0 Å |
| `AF_ipsae_at_best` | ipSAE for A_1 ↔ B_1 (0–1) | > 0.80 |
| `AG_best_min_pae` | Interface PAE: A_1 ↔ Cys113/122 context (C_1) in Å | < 5.0 Å |
| `AG_ipsae_at_best` | ipSAE for A_1 ↔ C_1 | > 0.80 |

Good binders should show **low PAE and high ipSAE with BOTH target chains** (B_1 and C_1) —
that means the scaffold confidently contacts all three Cys residues simultaneously, which
is required for proper Pb²⁺ coordination.

Note: `best_binder_plddt` includes the SpyTag residues. A moderate drop in pLDDT at the
C-terminal 19 residues is acceptable and expected for a flexible surface tag.

**Verify chain IDs from the first `.score` file before running — update below if needed.**

In [ ]:
# Expected chain IDs in RF3 output (order follows the template JSON components):
#   A_1 = scaffold + SpyTag  (our designed protein)
#   B_1 = chain C context    (residues 72–84, contains Cys78)
#   C_1 = chain D context    (residues 107–128, contains Cys113 and Cys122)
#   D_1 = Pb²⁺ ion           (HETATM from context PDB)
#
# If RF3 assigns different chain IDs, update these three variables:
BINDER   = "A_1"
TARGET_F = "B_1"   # primary interface:   scaffold ↔ Cys78 chain
TARGET_G = "C_1"   # secondary interface: scaffold ↔ Cys113/122 chain

OUT_CSV = f"{scores_dir}/rf3_gathered_metrics.csv"

df_rf3 = gather_rf3_metrics(
    parent=rf3_out_dir,
    binder=BINDER,
    target_f=TARGET_F,
    target_g=TARGET_G,
    out_csv=OUT_CSV,
)
print(f"{len(df_rf3)} designs scored")
if df_rf3.empty:
    print("  [HINT] If 0 designs scored, check that BINDER/TARGET_F/TARGET_G chain IDs")
    print("  match the chain IDs in your RF3 .score files. Use bcat or a text editor.")
df_rf3.head()

In [ ]:
# -- Scatter: interface PAE vs binder pLDDT, coloured by ipSAE -------------------
PAE_CUT   = 5.0    # AF_best_min_pae  < PAE_CUT   (Å)
PLDDT_CUT = 0.85   # best_binder_plddt > PLDDT_CUT (0-1)
IPSAE_CUT = 0.80   # AF_ipsae_at_best  > IPSAE_CUT (0-1)

df = df_rf3
x   = df['AF_best_min_pae'].to_numpy(float)
y   = df['best_binder_plddt'].to_numpy(float)
c   = df['AF_ipsae_at_best'].to_numpy(float)
mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(c)

plt.rcParams.update({
    "font.family": "sans-serif", "font.size": 13,
    "axes.linewidth": 1.1, "figure.dpi": 150,
})
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(x[mask], y[mask], c=c[mask], s=20, alpha=0.85,
                linewidths=0, cmap='viridis', vmin=0, vmax=1)
ax.axvline(PAE_CUT,   color='k', lw=0.9, ls='--', alpha=0.6)
ax.axhline(PLDDT_CUT, color='k', lw=0.9, ls='--', alpha=0.6)
ax.set_xlabel('Interface PAE (Å) — scaffold vs Cys78 context  [lower = better]')
ax.set_ylabel('Binder pLDDT  [higher = better]')
ax.set_title(f'RF3 validation — Pb-Cys3 scaffold campaign (n={len(df)})')
ax.set_xlim(left=0)
ax.set_ylim(0, 1.0)
ax.grid(True, linestyle=':', linewidth=0.6, alpha=0.5)
fig.colorbar(sc, ax=ax).set_label('ipSAE  [higher = better]')
fig.tight_layout()
plt.show()

In [ ]:
# -- Scatter: PAE vs ipSAE (primary interface: scaffold ↔ Cys78 context) ----------
x   = df['AF_best_min_pae'].to_numpy(float)
y   = df['AF_ipsae_at_best'].to_numpy(float)
c   = df['AF_ipsae_at_best'].to_numpy(float)
mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(c)

fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(x[mask], y[mask], c=c[mask], s=20, alpha=0.85,
                linewidths=0, cmap='viridis', vmin=0, vmax=1)
ax.axvline(PAE_CUT,   color='k', lw=0.9, ls='--', alpha=0.6)
ax.axhline(IPSAE_CUT, color='k', lw=0.9, ls='--', alpha=0.6)
ax.set_xlabel('min-PAE (Å)  [lower = better]')
ax.set_ylabel('inter-chain ipSAE  [higher = better]')
ax.set_title('Scaffold ↔ Cys78 context (A_1 vs B_1)')
ax.set_xlim(left=0)
ax.set_ylim(0, 1.0)
ax.grid(True, linestyle=':', linewidth=0.6, alpha=0.5)
fig.colorbar(sc, ax=ax).set_label('ipSAE')
fig.tight_layout()
plt.show()

In [ ]:
# -- Two-panel: PAE vs pLDDT  |  ipSAE vs pLDDT ----------------------------------
x_pae   = df['AF_best_min_pae'].to_numpy(float)
x_ipsae = df['AF_ipsae_at_best'].to_numpy(float)
y_plddt = df['best_binder_plddt'].to_numpy(float)
mask = np.isfinite(x_pae) & np.isfinite(x_ipsae) & np.isfinite(y_plddt)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, x_vals, xlabel, cutval in [
    (axes[0], x_pae,   'Interface PAE (Å)  [lower = better]', PAE_CUT),
    (axes[1], x_ipsae, 'ipSAE  [higher = better]',            IPSAE_CUT),
]:
    sc = ax.scatter(x_vals[mask], y_plddt[mask],
                    c=x_ipsae[mask], s=20, alpha=0.85, linewidths=0,
                    cmap='viridis', vmin=0, vmax=1)
    ax.axhline(PLDDT_CUT, color='k', lw=0.9, ls='--', alpha=0.6)
    ax.set_xlabel(xlabel)
    ax.set_ylim(0, 1.0)
    ax.grid(True, linestyle=':', linewidth=0.6, alpha=0.5)
    fig.colorbar(sc, ax=ax).set_label('ipSAE')

axes[0].axvline(PAE_CUT,   color='k', lw=0.9, ls='--', alpha=0.6)
axes[1].axvline(IPSAE_CUT, color='k', lw=0.9, ls='--', alpha=0.6)
axes[0].set_xlim(left=0)
axes[1].set_xlim(0, 1.0)
axes[0].set_ylabel('Binder pLDDT  [higher = better]')
axes[0].set_title('PAE vs pLDDT')
axes[1].set_title('ipSAE vs pLDDT')
fig.suptitle(f'RF3 validation — Pb-Cys3 scaffold + SpyTag ({len(df)} designs)', fontsize=13)
fig.tight_layout()
plt.show()

## 4. Collect your best binders

Designs passing all three thresholds (PAE < 5 Å, pLDDT > 0.85, ipSAE > 0.80) are
copied to `best_binders/` (their RF3 input JSON + CIF + scored CSV). These are your
Pb-binding scaffold candidates to inspect in PyMOL or ChimeraX.

**Selection considerations:**
- Check **both A–B and A–C interface scores** (`AF_*` and `AG_*`). Designs with high
  confidence on the Cys78 interface but low confidence on Cys113/122 (or vice versa)
  are risky — they may not properly engage all three Cys ligands for Pb coordination.
- **SpyTag pLDDT** — in the CIF structure, look at the B-factor (annotated with pLDDT)
  for the last ~19 residues. Low values (< 0.7) are desirable: they indicate the tag
  is flexible and accessible for SpyCatcher conjugation.
- **Scaffold compactness** — inspect in PyMOL. The scaffold should pack tightly around
  the three Cys residues without large voids.

In [ ]:
import gzip as _gz

pass_mask = (
    (df_rf3['AF_best_min_pae']   < PAE_CUT) &
    (df_rf3['best_binder_plddt'] > PLDDT_CUT) &
    (df_rf3['AF_ipsae_at_best']  > IPSAE_CUT)
)
df_best = df_rf3[pass_mask].copy()
print(f"{len(df_best)} / {len(df_rf3)} designs pass thresholds "
      f"(PAE<{PAE_CUT} Å, pLDDT>{PLDDT_CUT}, ipSAE>{IPSAE_CUT})")

best_dir = Path(best_binders_dir)
best_dir.mkdir(parents=True, exist_ok=True)

copied_cif, copied_json, missing = 0, 0, []
for _, row in df_best.iterrows():
    did  = row['design_id']
    bidx = int(row['best_batch_idx'])
    sf   = Path(row['score_file'])

    cif_src  = sf.parent / f"{did}_model_{bidx}.cif.gz"
    json_src = Path(rf3_json_dir) / f"{did}.json"

    dst_cif = best_dir / f"{did}_model_{bidx}.cif"   # decompress for easy viewing
    if cif_src.exists():
        with _gz.open(cif_src, "rb") as gz_in, open(dst_cif, "wb") as out:
            out.write(gz_in.read())
        copied_cif += 1
    else:
        missing.append(str(cif_src))

    dst_json = best_dir / f"{did}.json"
    if json_src.exists():
        shutil.copy2(json_src, dst_json)
        copied_json += 1
    else:
        missing.append(str(json_src))

print(f"Copied {copied_cif} CIF + {copied_json} JSON -> {best_dir}")
if missing:
    print(f"Missing {len(missing)} source file(s):")
    for m in missing[:10]:
        print("  ", m)

best_csv = best_dir / "best_binders_metrics.csv"
df_best.to_csv(best_csv, index=False)
print(f"Metrics -> {best_csv}")
df_best[["design_id", "AF_best_min_pae", "best_binder_plddt",
          "AF_ipsae_at_best", "AG_best_min_pae", "AG_ipsae_at_best"]]

## Optional: Tandem-repeat constructs for two Pb-binding sites

The simplest route to a two-site protein is **tandem duplication** of the best
scaffold. No additional RFD3 run is needed:

```
[scaffold] — GSGSGSG — [scaffold] — GSGSGSAHIVMVDAYKPTK
    site 1   7-aa linker   site 2       SpyTag (C-term)
```

Each copy of the scaffold contributes one complete Pb-Cys3 binding site. The resulting
protein (~200–300 aa total, plus SpyTag) has two independent Pb²⁺ coordination sites and
is anchored to the membrane via its single C-terminal SpyTag — exactly the geometry needed
for a high-avidity surface-displayed sensor or affinity column.

**Important:** these sequences are for expression and experimental testing. To validate
computationally, you would need to fold the tandem against two copies of the motif context
as separate RF3 components — a straightforward extension of the RF3 template, not shown here.

Run this cell only after `df_best` is non-empty.

In [ ]:
# -- Optional: generate tandem-repeat sequences from best binders ------------------

TANDEM_LINKER = "GSGSGSG"   # 7 aa; long enough to decouple the two sites

tandem_dir = Path(best_binders_dir) / "tandem_repeats"
tandem_dir.mkdir(parents=True, exist_ok=True)

tandem_records = []

for _, row in df_best.iterrows():
    did      = row['design_id']
    json_src = Path(rf3_json_dir) / f"{did}.json"
    if not json_src.exists():
        print(f"[SKIP] {did} — RF3 JSON not found")
        continue

    with open(json_src) as f:
        jdata = json.load(f)

    # Recover scaffold-only sequence by stripping the SpyTag appended in cell-21
    full_chain_a = next(
        c["seq"] for c in jdata["components"]
        if isinstance(c, dict) and c.get("chain_id") == "A"
    )
    scaffold_seq = full_chain_a[: len(full_chain_a) - len(SPYTAG)]

    # Build: site1 + GS-linker + site2 + SpyTag
    tandem_seq = scaffold_seq + TANDEM_LINKER + scaffold_seq + SPYTAG

    fasta_out = tandem_dir / f"{did}_tandem.fa"
    with open(fasta_out, "w") as f:
        f.write(f">{did}_tandem | two_Pb_sites | SpyTag_Cterm\n{tandem_seq}\n")

    tandem_records.append({
        "design_id":         did,
        "scaffold_len":      len(scaffold_seq),
        "tandem_total_len":  len(tandem_seq),
        "fasta":             str(fasta_out),
    })

df_tandem = pd.DataFrame(tandem_records)
print(f"Generated {len(df_tandem)} tandem-repeat FASTA(s) -> {tandem_dir}")
if not df_tandem.empty:
    print("\nNext steps:")
    print("  1. Order synthetic genes for the top 3–5 tandem constructs.")
    print("  2. Express in E. coli (His-tag or direct SpyCatcher-resin capture).")
    print("  3. Test Pb2+ binding by ITC or fluorescence quenching vs. Zn2+, Cd2+, Cu2+.")
    print("  4. For column: pack SpyCatcher-Sepharose, flow Pb2+-spiked buffer, measure ICP-MS.")
df_tandem